In [1]:
# Create training file from the active molecules
import pandas as pd
from pathlib import Path
import json

active_molecules = pd.read_csv("../data/active_compounds_mpo.smi", sep="\t", header=None, names=["SMILES", "target"])
# Get idx2target
idx2target = json.loads(Path("../data/target_conditions_to_index_mpo.json").read_text())
idx2target = idx2target["idx_to_gene"]

# Add target to active molecules
active_molecules["target_name"] = active_molecules["target"].map(
    lambda x: idx2target[str(x)]
)
active_molecules.sample(5)

,SMILES,target,target_name
2688,O=C1C(=Cc2ccc([N+](=O)[O-])cc2)CCc2ccccc21,358,SENP7
201852,CNc1nc(N)nc2nc(-c3ccccc3C(F)(F)F)ccc12,93,PTPN1
198711,COc1cccc2c1C(=O)c1c(O)c3c(c(O)c1C2=O)C[C@](O)(...,429,BBB
248723,O=C(Nc1ccc([N+](=O)[O-])cc1)N1CCN(c2ncccc2C(F)...,164,TRPV1
111870,Cc1cc2c(Nc3ccc4nc(N)sc4c3)c(C#N)cnc2cc1OCCCN1C...,272,SRC


In [2]:
conditions = {
    "Alzheimer's Disease": ["AChE", "MAOB"],
    "Schizophrenia": ["D2R", "_5HT2A"],
    "Parkinson's Disease": ["D2R", "D3R"],
}

for disease, targets in conditions.items():
    # Get SMILES where target_name is in targets
    smiles = active_molecules[active_molecules["target_name"].isin(targets)][["SMILES"]]
    smiles.to_csv(f"../data/training_files/active_compounds_{disease}.smi", header=False, index=False)

In [3]:
import sys
sys.path.append("..")
from benchmarks.guacamol.assess_distribution_learning import assess_distribution_learning_from_smiles


# Load generated molecules SMILES
for disease, targets in conditions.items():
    targets_file = "_".join(targets) + "_SUM.csv"
    smiles_list = pd.read_csv(f"../generated_molecules/25-epoch/{targets_file}")["SMILES"].tolist()
    assess_distribution_learning_from_smiles(
        smiles_list, 
        chembl_training_file=f"../data/training_files/active_compounds_{disease}.smi",
        json_output_file=f"../benchmarks/reports/distribution_learning_{disease}.json",
        number_samples=2000
    )

/home/arthurcerveira/miniconda3/envs/gnn2/lib/python3.12/site-packages/fcd/fcd.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_config = torch.load(model_path)
